In [ ]:
!pip install matplotlib
%pip install PyPDF2

In [ ]:
import sqlite3

def inicializar_bd_final():
    conexion = sqlite3.connect('clinica_local.db')
    cursor = conexion.cursor()
    
    # Limpiamos para la corrida final
    cursor.execute("DROP TABLE IF EXISTS pacientes")
    cursor.execute("DROP TABLE IF EXISTS registros_triage")

    cursor.execute('''
    CREATE TABLE pacientes (
        id_paciente INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_completo TEXT,
        curp TEXT,
        edad INTEGER,
        sexo TEXT
    )
    ''')

    cursor.execute('''
    CREATE TABLE registros_triage (
        id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
        id_paciente INTEGER,
        pa_sistolica INTEGER,
        pa_diastolica INTEGER,
        frec_cardiaca INTEGER,
        frec_respiratoria INTEGER,
        peso REAL,
        talla REAL,
        cintura REAL,
        glucosa INTEGER,
        temperatura REAL,
        spo2 INTEGER,
        condicion_glucosa TEXT,
        diagnosticos_previos TEXT,
        motivo_consulta TEXT,
        observaciones TEXT,
        nivel_alerta TEXT,
        justificacion_clinica TEXT,
        FOREIGN KEY(id_paciente) REFERENCES pacientes(id_paciente)
    )
    ''')

    conexion.commit()
    conexion.close()
    print("✅ Base de datos relacional lista para el prototipo final.")

inicializar_bd_final()

In [ ]:
import PyPDF2

ruta_pdf = "Historia-clinica.pdf"

texto_pdf = ""
try:
    with open(ruta_pdf, "rb") as archivo:
        lector_pdf = PyPDF2.PdfReader(archivo)
        for pagina in lector_pdf.pages:
            texto_pdf += pagina.extract_text() + "\n"
    print("✅ Texto extraído del PDF con éxito. Listo para Gemma 4.")
except Exception as e:
    print(f"❌ Error al leer el PDF: {e}. Revisa que el nombre del archivo sea exacto.")

In [ ]:
import requests
import json

url_local = "http://localhost:11434/api/generate"

prompt_maestro = f"""
Eres un motor de inferencia clínica y estructuración de datos.
Analiza este texto extraído de un expediente médico:

"{texto_pdf}"

TAREAS:
1. EXTRACCIÓN Y LIMPIEZA: Extrae los datos y normalízalos. Si un dato no existe en el texto, asigna estrictamente null (sin comillas). Los números deben ser formato numérico.
2. INFERENCIA CLÍNICA: Analiza los signos vitales, motivos de consulta y observaciones. Evalúa el riesgo metabólico o de urgencia. Define el "nivel_alerta" como ALTA, MEDIA o BAJA, y redacta una "justificacion_clinica" breve de tu decisión.

Devuelve ÚNICAMENTE un objeto JSON válido con esta estructura exacta:
{{
  "nombre_completo": null,
  "curp": null,
  "edad": null,
  "sexo": null,
  "pa_sistolica": null,
  "pa_diastolica": null,
  "frec_cardiaca": null,
  "frec_respiratoria": null,
  "peso": null,
  "talla": null,
  "cintura": null,
  "glucosa": null,
  "temperatura": null,
  "spo2": null,
  "condicion_glucosa": null,
  "diagnosticos_previos": null,
  "motivo_consulta": null,
  "observaciones": null,
  "nivel_alerta": null,
  "justificacion_clinica": null
}}
"""

payload = {
    "model": "gemma", 
    "prompt": prompt_maestro,
    "stream": False,
    "format": "json"
}

print("🧠 Gemma 4 estructurando datos y calculando alertas clínicas...")
respuesta = requests.post(url_local, json=payload)
resultado_crudo = respuesta.json().get("response", "")
print("✅ JSON generado exitosamente.")

In [ ]:
if resultado_crudo:
    try:
        texto_json = resultado_crudo.strip()
        if texto_json.startswith("```json"):
            texto_json = texto_json[7:-3].strip()
        elif texto_json.startswith("```"):
            texto_json = texto_json[3:-3].strip()
            
        datos = json.loads(texto_json)
        
        conexion = sqlite3.connect('clinica_local.db')
        cursor = conexion.cursor()
        
        # Insertar Paciente
        cursor.execute('''INSERT INTO pacientes (nombre_completo, curp, edad, sexo) 
                          VALUES (?, ?, ?, ?)''', 
                       (datos.get('nombre_completo'), datos.get('curp'), 
                        datos.get('edad'), datos.get('sexo')))
        id_pac = cursor.lastrowid
        
        # Insertar Historial y Análisis
        cursor.execute('''
            INSERT INTO registros_triage (
                id_paciente, pa_sistolica, pa_diastolica, frec_cardiaca, frec_respiratoria,
                peso, talla, cintura, glucosa, temperatura, spo2, condicion_glucosa, 
                diagnosticos_previos, motivo_consulta, observaciones, nivel_alerta, justificacion_clinica
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            id_pac, datos.get('pa_sistolica'), datos.get('pa_diastolica'), datos.get('frec_cardiaca'),
            datos.get('frec_respiratoria'), datos.get('peso'), datos.get('talla'), datos.get('cintura'),
            datos.get('glucosa'), datos.get('temperatura'), datos.get('spo2'), datos.get('condicion_glucosa'),
            datos.get('diagnosticos_previos'), datos.get('motivo_consulta'), datos.get('observaciones'),
            datos.get('nivel_alerta'), datos.get('justificacion_clinica')
        ))
        
        conexion.commit()
        conexion.close()
        print("✅ Expediente y análisis clínico guardados en SQLite.")
        
    except Exception as e:
        print(f"❌ Error al procesar o guardar los datos: {e}")

In [ ]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('clinica_local.db')
df_pacientes = pd.read_sql_query("SELECT * FROM pacientes", conexion)
df_registros = pd.read_sql_query("SELECT * FROM registros_triage", conexion)
conexion.close()

print("👤 === BASE DE DATOS: PACIENTES ===")
display(df_pacientes)

print("\n🏥 === MOTOR DE INFERENCIA CLÍNICA (GEMMA 4) ===")
# Mostrar específicamente la magia de la IA
display(df_registros[['id_paciente', 'glucosa', 'motivo_consulta', 'observaciones', 'nivel_alerta', 'justificacion_clinica']])